In [1]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, precision_score,
                              recall_score, f1_score, accuracy_score)
import shap

df = pd.read_csv('../data/processed/02_features.csv')

# ── FEATURES ──
FEATURES = [
    'Price',
    'shelf_life_days',
    'warranty_enc',
    'category_enc',
    'Stock Quantity',
    'overhang_ratio'
]

TARGET = 'is_at_risk'

X = df[FEATURES]
y = df[TARGET]

# ── SPLIT ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# ── BASELINE: Logistic Regression ──
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:,1]

lr_metrics = {
    'AUC-ROC':   round(roc_auc_score(y_test, lr_prob), 4),
    'Precision': round(precision_score(y_test, lr_pred), 4),
    'Recall':    round(recall_score(y_test, lr_pred), 4),
    'F1-Score':  round(f1_score(y_test, lr_pred), 4),
    'Accuracy':  round(accuracy_score(y_test, lr_pred), 4),
}
print("\nLogistic Regression:", lr_metrics)

# ── MAIN MODEL: Random Forest ──
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:,1]

rf_metrics = {
    'AUC-ROC':   round(roc_auc_score(y_test, rf_prob), 4),
    'Precision': round(precision_score(y_test, rf_pred), 4),
    'Recall':    round(recall_score(y_test, rf_pred), 4),
    'F1-Score':  round(f1_score(y_test, rf_pred), 4),
    'Accuracy':  round(accuracy_score(y_test, rf_pred), 4),
}
print("Random Forest:", rf_metrics)

# ── SHAP FEATURE IMPORTANCE ──
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

shap_importance = pd.DataFrame({
    'feature': FEATURES,
    'importance': np.abs(shap_values[1]).mean(axis=0)
}).sort_values('importance', ascending=False)

shap_importance['pct'] = (
    shap_importance['importance'] / shap_importance['importance'].sum() * 100
).round(1)

print("\nTop 10 features:")
print(shap_importance.head(10))

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values[1], X_test, feature_names=FEATURES,
                  show=False, plot_type='bar')
plt.tight_layout()
plt.savefig('../reports/figures/05_shap_importance.png', bbox_inches='tight')
plt.close()
print("SHAP plot saved.")

# ── DASHBOARD EXPORT ──

# safer risk breakdown (handles missing categories)
risk_breakdown = df['Risk_Level'].value_counts().reindex(
    ['Critical','High','Medium','Low'], fill_value=0
).to_dict()

dashboard_data = {
    'kpis': {
        'total_skus': len(df),
        'avg_stock_qty': round(df['Stock Quantity'].mean(), 1),

        # FIXED: use days_to_expiry instead of Days_to_Expiry
        'dual_risk_count': int(df[
            (df['Stock Quantity'] > 75) & (df['days_to_expiry'] < 180)
        ].shape[0]),

        'capital_at_risk': round(
            df[df['is_at_risk']==1]['financial_exposure'].sum(), 0
        ),

        # FIXED: <= 0 instead of == 0
        'write_off_rate': round(
            (df['days_to_expiry'] <= 0).mean() * 100, 1
        )
    },

    'risk_breakdown': risk_breakdown,

    'model_logistic': lr_metrics,
    'model_rf': rf_metrics,

    'feature_importance': shap_importance.head(10)[['feature','pct']].to_dict('records'),

    'stock_distribution': df['stock_bucket'].value_counts().sort_index().to_dict(),
    'expiry_distribution': df['expiry_bucket'].value_counts().sort_index().to_dict(),

    'price_by_category': df.groupby('Product Category')['Price'].mean().round(2).to_dict(),
}

with open('../outputs/dashboard_data.json', 'w') as f:
    json.dump(dashboard_data, f, indent=2)

print("\ndashboard_data.json saved to outputs/")


Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)


Train: (8000, 6), Test: (2000, 6)

Logistic Regression: {'AUC-ROC': 0.9293, 'Precision': 0.7761, 'Recall': 0.7028, 'F1-Score': 0.7376, 'Accuracy': 0.8755}
Random Forest: {'AUC-ROC': 0.9998, 'Precision': 0.9959, 'Recall': 0.9859, 'F1-Score': 0.9909, 'Accuracy': 0.9955}

Top 10 features:
           feature  importance   pct
1  shelf_life_days    0.192433  38.1
0            Price    0.147270  29.2
4   Stock Quantity    0.114043  22.6
5   overhang_ratio    0.046360   9.2
3     category_enc    0.002490   0.5
2     warranty_enc    0.002020   0.4
SHAP plot saved.

dashboard_data.json saved to outputs/


The figure layout has changed to tight


In [2]:
print(df['is_at_risk'].value_counts())
print(df['is_at_risk'].value_counts(normalize=True))

is_at_risk
0    7510
1    2490
Name: count, dtype: int64
is_at_risk
0    0.751
1    0.249
Name: proportion, dtype: float64


In [6]:
print("=== PASTE THIS TO CLAUDE ===")
print(f"Total SKUs: {len(df)}")
print(f"Total Portfolio Value: ${df['inventory_value'].sum():,.0f}")
print(f"At-Risk SKUs: {df['is_at_risk'].sum()} ({df['is_at_risk'].mean():.1%})")
print(f"At-Risk Value: ${df[df['is_at_risk']==1]['inventory_value'].sum():,.0f}")
print(f"Safe SKUs: {(df['is_at_risk']==0).sum()} ({(df['is_at_risk']==0).mean():.1%})")
print(f"Safe Value: ${df[df['is_at_risk']==0]['inventory_value'].sum():,.0f}")
print(f"High Stock SKUs (>75): {(df['Stock Quantity']>75).sum()}")
print(f"High Stock Value: ${df[df['Stock Quantity']>75]['inventory_value'].sum():,.0f}")
print(f"Short Shelf Life SKUs (pct_remaining < 0.45): {(df['pct_shelf_life_remaining']<0.45).sum()}")
print()
print("=== CATEGORY BREAKDOWN ===")
cat = df.groupby('Product Category').agg(
    SKUs=('Price','count'),
    portfolio_value=('inventory_value','sum'),
    at_risk=('is_at_risk','sum'),
    high_stock=('Stock Quantity', lambda x: (x>75).sum()),
).assign(
    at_risk_pct=lambda x: (x['at_risk']/x['SKUs']*100).round(1),
    overstock_pct=lambda x: (x['high_stock']/x['SKUs']*100).round(1)
)
print(cat.to_string())
print()
print("=== RISK LEVEL COUNTS ===")
print(df['Risk_Level'].value_counts())
print()
print("=== LR METRICS ===")
print(lr_metrics)
print("=== RF METRICS ===")
print(rf_metrics)
print()
print("=== FEATURE IMPORTANCE ===")
print(shap_importance.to_string())

=== PASTE THIS TO CLAUDE ===
Total SKUs: 10000
Total Portfolio Value: $128,832,423
At-Risk SKUs: 2490 (24.9%)
At-Risk Value: $56,722,607
Safe SKUs: 7510 (75.1%)
Safe Value: $72,109,816
High Stock SKUs (>75): 2530
High Stock Value: $56,718,863
Short Shelf Life SKUs (pct_remaining < 0.45): 3320

=== CATEGORY BREAKDOWN ===
                  SKUs  portfolio_value  at_risk  high_stock  at_risk_pct  overstock_pct
Product Category                                                                        
Clothing          3341      42557374.02      836         837         25.0           25.1
Electronics       3361      43196605.81      806         852         24.0           25.3
Home Appliances   3298      43078443.32      848         841         25.7           25.5

=== RISK LEVEL COUNTS ===
Risk_Level
Low         4991
Medium      3326
High         842
Critical     841
Name: count, dtype: int64

=== LR METRICS ===
{'AUC-ROC': 0.9293, 'Precision': 0.7761, 'Recall': 0.7028, 'F1-Score': 0.7376, 'A